In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data/raw/banking77"
cache_dir = project_root / "data/processed/lab10"

raw_dir.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)

source_url = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/"
    "master/banking_data/train.csv"
)
raw_path = raw_dir / "train.csv"

if not raw_path.exists():
    urlretrieve(source_url, raw_path)

questions = pd.read_csv(raw_path)
questions.insert(0, "question_id", range(len(questions)))

assert questions[["text", "category"]].notna().all().all()

print(f"Questions: {len(questions):,}")
print(f"Categories: {questions['category'].nunique()}")
display(questions.head())

Questions: 10,003
Categories: 77


,question_id,text,category
0,0,I am still waiting on my card?,card_arrival
1,1,What can I do if my card still hasn't arrived ...,card_arrival
2,2,I have been waiting over a week. Is the card s...,card_arrival
3,3,Can I track my card while it is in the process...,card_arrival
4,4,"How do I know if I will get my card, or if it ...",card_arrival


In [2]:
selected_categories = [
    "cancel_transfer",
    "pending_transfer",
    "failed_transfer",
    "declined_transfer",
    "transfer_not_received_by_recipient",
    "balance_not_updated_after_bank_transfer",
    "card_payment_not_recognised",
    "pending_card_payment",
    "transaction_charged_twice",
    "request_refund",
]

assert set(selected_categories).issubset(
    set(questions["category"])
)

selected = questions.loc[
    questions["category"].isin(selected_categories)
].copy()

# A comparison key for finding repeated text.
# Preserve the original text for the model and for inspection.
selected["text_key"] = (
    selected["text"]
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

assert selected["text_key"].ne("").all()

# Exclude identical normalized texts with conflicting labels.
conflicting = (
    selected.groupby("text_key")["category"]
    .transform("nunique")
    .gt(1)
)

before = len(selected)

selected = (
    selected.loc[~conflicting]
    .drop_duplicates("text_key")
    .reset_index(drop=True)
)

print(f"Selected questions: {len(selected):,}")
print(f"Rows excluded during duplicate checks: {before - len(selected)}")

display(selected["category"].value_counts().sort_index())

Selected questions: 1,588
Rows excluded during duplicate checks: 0


category
balance_not_updated_after_bank_transfer    171
cancel_transfer                            157
card_payment_not_recognised                168
declined_transfer                          133
failed_transfer                            137
pending_card_payment                       159
pending_transfer                           148
request_refund                             169
transaction_charged_twice                  175
transfer_not_received_by_recipient         171
Name: count, dtype: int64

In [6]:
''' 
Why keep them separate? If the query itself were also in the corpus, 
the search could simply find that identical question. 
We want to test whether it can find other questions with a similar meaning.
'''
corpus, queries = train_test_split(
    selected,
    test_size=0.20,
    stratify=selected["category"],
    random_state=42,
)

corpus = corpus.reset_index(drop=True)
queries = queries.reset_index(drop=True)

assert set(corpus["question_id"]).isdisjoint(
    set(queries["question_id"])
)
assert set(corpus["text_key"]).isdisjoint(
    set(queries["text_key"])
)
assert set(corpus["category"]) == set(queries["category"])

split_summary = pd.DataFrame({
    "corpus": corpus["category"].value_counts(),
    "queries": queries["category"].value_counts(),
}).sort_index()

display(split_summary)

display(
    queries[["question_id", "text", "category"]]
    .sample(5, random_state=42)
)

,corpus,queries
category,,
balance_not_updated_after_bank_transfer,137,34
cancel_transfer,126,31
card_payment_not_recognised,134,34
declined_transfer,106,27
failed_transfer,110,27
pending_card_payment,127,32
pending_transfer,118,30
request_refund,135,34
transaction_charged_twice,140,35


,question_id,text,category
73,8701,How long do transfers from a UK bank account u...,balance_not_updated_after_bank_transfer
277,6340,I have been charged twice this week for a coup...,transaction_charged_twice
25,7485,Why can't I transfer my money into someone els...,failed_transfer
256,5548,why was a transfer declined,declined_transfer
9,5466,It has been forever ago that I made an order o...,request_refund


In [7]:
corpus.to_parquet(
    cache_dir / "corpus.parquet",
    index=False,
)
queries.to_parquet(
    cache_dir / "queries.parquet",
    index=False,
)

print(f"Searchable questions: {len(corpus):,}")
print(f"Evaluation queries: {len(queries):,}")

Searchable questions: 1,270
Evaluation queries: 318


In [8]:
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    dtype=np.float32,
)

corpus_tfidf = vectorizer.fit_transform(corpus["text"])
query_tfidf = vectorizer.transform(queries["text"])

tfidf_scores = cosine_similarity(
    query_tfidf,
    corpus_tfidf,
)

print("Corpus vectors:", corpus_tfidf.shape)
print("Query vectors:", query_tfidf.shape)
print("Similarity scores:", tfidf_scores.shape)

Corpus vectors: (1270, 6536)
Query vectors: (318, 6536)
Similarity scores: (318, 1270)


In [9]:
import torch
from sentence_transformers import SentenceTransformer

torch.set_num_threads(2)

model_id = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(
    model_id,
    device="cpu",
)

/Users/sara/Projects/data-science-lab/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1116 21:34:37.274000 12906 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12666.04it/s]


In [12]:
example_text = queries.iloc[0]["text"]

token_ids = embedder.tokenizer(example_text)["input_ids"]
tokens = embedder.tokenizer.convert_ids_to_tokens(token_ids)

print("Original text:", example_text)
print("Tokens:", tokens)

Original text: Why isn't my balance reflecting a transfer I did?
Tokens: ['[CLS]', 'why', 'isn', "'", 't', 'my', 'balance', 'reflecting', 'a', 'transfer', 'i', 'did', '?', '[SEP]']


In [13]:
corpus_embeddings = embedder.encode(
    corpus["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

query_embeddings = embedder.encode(
    queries["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Corpus embeddings:", corpus_embeddings.shape)
print("Query embeddings:", query_embeddings.shape)

Batches: 100%|██████████| 10/10 [00:00<00:00, 12.55it/s]

Corpus embeddings: (1270, 384)
Query embeddings: (318, 384)


In [14]:
np.testing.assert_allclose(
    np.linalg.norm(corpus_embeddings, axis=1),
    1.0,
    atol=1e-5,
)
np.testing.assert_allclose(
    np.linalg.norm(query_embeddings, axis=1),
    1.0,
    atol=1e-5,
)

embedding_scores = query_embeddings @ corpus_embeddings.T

assert embedding_scores.shape == tfidf_scores.shape
assert np.isfinite(embedding_scores).all()

print("Embedding similarity scores:", embedding_scores.shape)

Embedding similarity scores: (318, 1270)


In [15]:
def show_matches(
    query_position,
    scores,
    corpus,
    queries,
    k=5,
):
    """Return the highest-scoring corpus questions for one query."""
    query_scores = scores[query_position]

    # Descending scores; preserve corpus order when scores tie.
    top_positions = np.argsort(
        -query_scores,
        kind="stable",
    )[:k]

    matches = corpus.iloc[top_positions][
        ["question_id", "text", "category"]
    ].copy()

    matches.insert(0, "rank", np.arange(1, len(matches) + 1))
    matches["similarity"] = query_scores[top_positions]

    # Labels are added after retrieval, only for inspection.
    matches["same_category"] = matches["category"].eq(
        queries.iloc[query_position]["category"]
    )

    return matches.reset_index(drop=True)

In [16]:
query_position = 0

print("Query:", queries.iloc[query_position]["text"])
print("Category:", queries.iloc[query_position]["category"])

print("\nTF-IDF results:")
display(
    show_matches(
        query_position,
        tfidf_scores,
        corpus,
        queries,
    )
)

print("\nEmbedding results:")
display(
    show_matches(
        query_position,
        embedding_scores,
        corpus,
        queries,
    )
)

Query: Why isn't my balance reflecting a transfer I did?
Category: balance_not_updated_after_bank_transfer

TF-IDF results:


,rank,question_id,text,category,similarity,same_category
0,1,6051,Why isn't my pending transfer finished?,pending_transfer,0.500097,False
1,2,8648,Why isn't the transfer to my account showing up?,balance_not_updated_after_bank_transfer,0.303034,True
2,3,8727,Why isn't the transferred money there?,balance_not_updated_after_bank_transfer,0.287142,True
3,4,8622,The transfer I made isn't reflected in my balance,balance_not_updated_after_bank_transfer,0.258738,True
4,5,8620,Why is my balance the same after a transfer?,balance_not_updated_after_bank_transfer,0.223223,True



Embedding results:


,rank,question_id,text,category,similarity,same_category
0,1,8622,The transfer I made isn't reflected in my balance,balance_not_updated_after_bank_transfer,0.909868,True
1,2,8620,Why is my balance the same after a transfer?,balance_not_updated_after_bank_transfer,0.885201,True
2,3,8674,I transfered my balance a while back. Why doe...,balance_not_updated_after_bank_transfer,0.845930,True
3,4,8653,My balance doesn't include my latest transfer,balance_not_updated_after_bank_transfer,0.821829,True
4,5,8673,I made a transfer but my balance hasn't been u...,balance_not_updated_after_bank_transfer,0.785934,True


In [ ]:
import joblib

arrays = {
    "corpus_embeddings": corpus_embeddings,
    "query_embeddings": query_embeddings,
    "tfidf_scores": tfidf_scores,
    "embedding_scores": embedding_scores,
    "corpus_question_ids": corpus["question_id"].to_numpy(),
    "query_question_ids": queries["question_id"].to_numpy(),
}

for name, values in arrays.items():
    np.save(cache_dir / f"{name}.npy", values)

joblib.dump(
    vectorizer,
    cache_dir / "tfidf_vectorizer.joblib",
)